# **Carga de datos**

Para este entrenamiento utilizaré un dataset de imágenes de retinografías para detectar el porcentaje de glaucoma. Para ello, he dividido los datos de la siguiente manera:

* **70%** para el entrenamiento
* **10%** para la validación
* **20%** para los tests

In [10]:
import os

raiz = 'Ojos'

# Datos de entrenamiento
train_dir = os.path.join(raiz, 'train')
train_dir_normal = os.path.join(train_dir, 'Normales')
train_dir_glaucoma = os.path.join(train_dir, 'Glaucomas')

# Datos de validación
val_dir = os.path.join(raiz, 'val')
val_dir_normal = os.path.join(val_dir, 'Normales')
val_dir_glaucoma = os.path.join(val_dir, 'Glaucomas')

# Datos de prueba
test_dir = os.path.join(raiz, 'test')
test_dir_normal = os.path.join(test_dir, 'Normales')
test_dir_glaucoma = os.path.join(test_dir, 'Glaucomas')

In [15]:
# Muestro la cantidad de imágenes en cada conjunto
suma_entrenamiento = len(os.listdir(train_dir_normal)) + len(os.listdir(train_dir_glaucoma))
suma_validacion = len(os.listdir(val_dir_normal)) + len(os.listdir(val_dir_glaucoma))
suma_prueba = len(os.listdir(test_dir_normal)) + len(os.listdir(test_dir_glaucoma))

print("Datos de entrenamiento:", suma_entrenamiento, "--- Normales:", len(os.listdir(train_dir_normal)), "--- Glaucoma:", len(os.listdir(train_dir_glaucoma)))
print("Datos de validación:", suma_validacion, "--- Normales:", len(os.listdir(val_dir_normal)), "--- Glaucoma:", len(os.listdir(val_dir_glaucoma)))
print("Datos de prueba:", suma_prueba, "--- Normales:", len(os.listdir(test_dir_normal)), "--- Glaucoma:", len(os.listdir(test_dir_glaucoma)))

Datos de entrenamiento: 339 --- Normales: 219 --- Glaucoma: 120
Datos de validación: 48 --- Normales: 31 --- Glaucoma: 17
Datos de prueba: 98 --- Normales: 63 --- Glaucoma: 35


# **Data Augmentation**

Para esta parte, voy a normalizar las imágenes y a hacer uso de la herramienta ImageDataGenerator del módulo *tensorflow.keras.preprocessing.image* para adaptar las imágenes a la red neuronal que voy a utilizar.

In [16]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg19 import preprocess_input


train_generator = ImageDataGenerator(rescale=1./255,
                                     preprocessing_function=preprocess_input,
                                     rotation_range=10,         # Rotaciones leves ±10°
                                     width_shift_range=0.05,    # Desplazamiento horizontal pequeño
                                     height_shift_range=0.05,   # Desplazamiento vertical pequeño
                                     zoom_range=0.1,            # Zoom leve
                                     horizontal_flip=True,      # Flip horizontal solo si anatómicamente válido
                                     brightness_range=[0.9, 1.1], # Variaciones leves de brillo
                                     fill_mode='nearest'
)
val_generator = ImageDataGenerator(rescale=1./255)
test_generator = ImageDataGenerator(rescale=1./255)

In [20]:
train_generator.flow_from_directory(train_dir,
                                    target_size=(224, 224),
                                    class_mode='binary',
                                    batch_size=16)

val_generator.flow_from_directory(val_dir,
                                  target_size=(224, 224),
                                  class_mode='binary',
                                  batch_size=16)

test_generator.flow_from_directory(test_dir,
                                   target_size=(224, 224),
                                   class_mode='binary',
                                   batch_size=16)

Found 339 images belonging to 2 classes.
Found 48 images belonging to 2 classes.
Found 98 images belonging to 2 classes.


# **Obtención del modelo (VGG19)**

Ya cargado y normalizado los datos previamente, ahora voy a descargar el modelo VGG19 para utilizarlo para mi entrenamiento y prueba, las imágenes ya han sido adaptadas a este modelo por lo que no hace falta modificar nada más de las mismas.